In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
import csv
# Increase the field size limit to accommodate large text fields (e.g., 1MB)
csv.field_size_limit(1048576)

#Data Preprocessing
#Loading the dataset to a Pandas Dataframe
news_dataset=pd.read_csv("/content/train.csv", engine='python', on_bad_lines='skip')
news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [ ]:
news_dataset.shape

(20128, 5)

In [ ]:
news_dataset.isnull().sum()

,0
id,0
title,543
author,1904
text,39
label,0


In [ ]:
#replacing the null values with empty string
news_dataset=news_dataset.fillna("")

In [ ]:
news_dataset.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [ ]:
#Merging the author name and news title
news_dataset['content']=news_dataset['author']+' '+news_dataset['title']

In [ ]:
print(news_dataset['content'])

0        Darrell Lucus House Dem Aide: We Didn’t Even S...
1        Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2        Consortiumnews.com Why the Truth Might Get You...
3        Jessica Purkiss 15 Civilians Killed In Single ...
4        Howard Portnoy Iranian woman jailed for fictio...
                               ...                        
20123    Lee Rogers Rigging the Election – Video IV: De...
20124    Henry Wolff U-M’s New ‘Chief Diversity Officer...
20125    Robert D. McFadden Zsa Zsa Gabor, Actress Famo...
20126    Vincent M. Mallozzi Pursuing a Dream in Film, ...
20127    Guest Author #TrumpProtest: Communists Mobiliz...
Name: content, Length: 20128, dtype: object


In [ ]:
#separating the data and label
X=news_dataset.drop(columns='label',axis=1)
Y=news_dataset['label']

In [ ]:
print(X)
print(Y)

          id  ...                                            content
0          0  ...  Darrell Lucus House Dem Aide: We Didn’t Even S...
1          1  ...  Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2          2  ...  Consortiumnews.com Why the Truth Might Get You...
3          3  ...  Jessica Purkiss 15 Civilians Killed In Single ...
4          4  ...  Howard Portnoy Iranian woman jailed for fictio...
...      ...  ...                                                ...
20123  20123  ...  Lee Rogers Rigging the Election – Video IV: De...
20124  20124  ...  Henry Wolff U-M’s New ‘Chief Diversity Officer...
20125  20125  ...  Robert D. McFadden Zsa Zsa Gabor, Actress Famo...
20126  20126  ...  Vincent M. Mallozzi Pursuing a Dream in Film, ...
20127  20127  ...  Guest Author #TrumpProtest: Communists Mobiliz...

[20128 rows x 5 columns]
0        1
1        0
2        1
3        1
4        1
        ..
20123    1
20124    1
20125    0
20126    0
20127    1
Name: label, Length: 2012

In [ ]:
#Stemming
#Stemming is the process of reducing a word to its root word(actor,acting->act)
port_stem=PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content=re.sub('[^a-zA-Z]',' ',content)
  stemmed_content=stemmed_content.lower()
  stemmed_content=stemmed_content.split()
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content=' '.join(stemmed_content)
  return stemmed_content


In [ ]:
news_dataset['content']=news_dataset['content'].apply(stemming)


In [ ]:
#Separating the data and label
X=news_dataset['content'].values
Y=news_dataset['label'].values

In [ ]:
print(X)

['darrel lucu hous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynn flynn hillari clinton big woman campu breitbart'
 'consortiumnew com truth might get fire' ...
 'robert mcfadden zsa zsa gabor actress famou glamour marriag die new york time'
 'vincent mallozzi pursu dream film tip spike lee new york time'
 'guest author trumpprotest communist mobil disrupt presid elect trump inaugur']


In [ ]:
print(Y)

[1 0 1 ... 0 0 1]


In [ ]:
Y.shape

(20128,)

In [ ]:
#converting textual data to numerical data
vectorizer=TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

In [ ]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 203837 stored elements and shape (20128, 16900)>
  Coords	Values
  (0, 264)	0.2704788375262788
  (0, 2457)	0.3707652711485457
  (0, 2930)	0.2475047363148132
  (0, 3564)	0.35852307110544857
  (0, 3752)	0.2704788375262788
  (0, 4909)	0.23285194343725513
  (0, 6921)	0.22001263699280094
  (0, 7600)	0.24829524726436045
  (0, 8523)	0.29147672671841424
  (0, 8794)	0.36222580681662475
  (0, 13300)	0.2561603907959716
  (0, 15485)	0.28347423627127083
  (1, 1483)	0.2936954352758167
  (1, 1875)	0.15549366627062566
  (1, 2198)	0.38391373223939185
  (1, 2784)	0.19148534472815795
  (1, 3533)	0.2646682365294974
  (1, 5435)	0.7127812157181822
  (1, 6734)	0.19082298708140252
  (1, 16575)	0.3016431180936072
  (2, 2914)	0.317946837483197
  (2, 3073)	0.46278475874192765
  (2, 5321)	0.3865764958325316
  (2, 5895)	0.3464668296861228
  (2, 9498)	0.49248503163144053
  :	:
  (20125, 12676)	0.17335913983882878
  (20125, 15097)	0.06912698044539282
  (2

In [ ]:
#splitting data into train and test
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=2)

In [ ]:
print(X.shape,X_train.shape,X_test.shape)

(20128, 16900) (16102, 16900) (4026, 16900)


In [ ]:
#Training the model:Logistic Regression
model=LogisticRegression()
model.fit(X_train,Y_train)

LogisticRegression()

In [ ]:
#Evaluation
#On training data
X_train_pred=model.predict(X_train)
train_data_acc=accuracy_score(Y_train,X_train_pred)
print("Accuracy on training data: ",train_data_acc)


Accuracy on training data:  0.986647621413489


In [ ]:
#On test data
X_test_pred=model.predict(X_test)
test_data_acc=accuracy_score(Y_test,X_test_pred)
print("Accuracy on testing data: ",test_data_acc)

Accuracy on testing data:  0.9773969200198709


In [ ]:
#Making a Predictive System
X_new=X_test[1]

prediction=model.predict(X_new)
print(prediction)

if(prediction[0]==0):
  print("The News is Real")
else:
  print("The News is Fake")

[1]
The News is Fake


In [ ]:
print(Y_test[1])

1
